# Prepping Data Challenge - Week 17

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [116]:
MONTHS = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']

In [44]:
budget = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_17_Data.xlsx', 'Budget')
months = [pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_17_Data.xlsx', f'Month-{MONTH}') for MONTH in
                    MONTHS]

In [57]:
budget = budget.dropna()
budget = budget.set_index(2023)

In [45]:
for i in range(len(months)):
    if 'Unnamed: 0' in months[i].columns:
        months[i] = months[i].drop('Unnamed: 0', axis=1)
    months[i].index = [months[i].columns[0]]
    months[i] = months[i].drop(months[i].columns[0], axis=1)

In [46]:
months = pd.concat(months)

In [58]:
budget

,Budget
2023,
Rent,1008000 SameAsLastYear
Utilities,120000 Appx
Security,90000 PaidUpfront
Wages,270000
Inventory,2300000 RoughEstimate
Transportation,180000 Estimate
Transaction Fees,54000 Appx
Advertising,300000 SameAsLastYear
Insurance,72000 MoreThanLastYear


In [59]:
months

,Rent,Utilities,Security,Wages,Inventory,Transportation,TransactionFees,Advertising,Insurance
2023-01-01,84000,9255.251667,7500,22500,341902.8658,21022.77917,5640.43,25574.99667,6000
2023-02-01,84000,9255.251667,7500,22500,211345.1858,11957.58917,5471.10,25574.99667,6000
2023-03-01,84000,9255.251667,7500,22500,230988.6658,18856.66917,4968.78,25574.99667,6000
2023-04-01,84000,9255.251667,7500,22500,263154.2458,16505.27917,5098.87,25574.99667,6000
2023-05-01,84000,9255.251667,7500,22500,234528.5758,17847.11917,5128.37,25574.99667,6000
2023-06-01,84000,9255.251667,7500,22500,229251.0358,18902.92917,5043.07,25574.99667,6000
2023-07-01,84000,9255.251667,7500,22500,236123.0458,16899.54917,4856.25,25574.99667,6000
2023-08-01,84000,9255.251667,7500,22500,206210.8058,17955.35917,4770.95,25574.99667,6000
2023-09-01,84000,9255.251667,7500,22500,213082.8158,19297.19917,4800.45,25574.99667,6000
2023-10-01,84000,9255.251667,7500,22500,207805.2758,16945.80917,4930.54,25574.99667,6000


### Change Budget column to numerical

In [65]:
budget['Budget'] = budget['Budget'].apply(lambda x: int(str(x).split(' ')[0]))

### Join the columns

#### Filter out areas where annual spending was under budget

In [75]:
months.columns = ['Rent', 'Utilities', 'Security', 'Wages', 'Inventory', 'Transportation', 'Transaction Fees', 'Advertising', 'Insurance']

In [93]:
overspending_areas = budget.iloc[np.where((months.sum() - (budget['Budget']) > 0))].index

In [112]:
months_over = months[overspending_areas].round(2)

#### Join the budget

In [119]:
spending_vs_budget = pd.concat([months_over, (budget.T[overspending_areas] / 12).round(2)]).T

In [123]:
spending_vs_budget.columns = MONTHS + ['Budget']

### Find the area of most overspending

In [128]:
for month in MONTHS:
    spending_vs_budget[f'{month}:Overspend'] = spending_vs_budget[month] - spending_vs_budget['Budget']

In [134]:
months.index = MONTHS

In [136]:
months['Category'] = spending_vs_budget.idxmax()[-12:].values

In [170]:
months['Actual Spending'] = months.to_numpy()[range(len(months)), months.columns.get_indexer(months['Category'])]

In [178]:
months['Actual Spending'] = months['Actual Spending'].astype(float).round().astype(int)

In [181]:
budget

,Budget
2023,
Rent,1008000
Utilities,120000
Security,90000
Wages,270000
Inventory,2300000
Transportation,180000
Transaction Fees,54000
Advertising,300000
Insurance,72000


In [189]:
months['Budget'] = months['Category'].apply(lambda x: int(round(budget.loc[x] / 12)))

C:\Users\tobia\AppData\Local\Temp\ipykernel_25348\1926832958.py:1: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  months['Budget'] = months['Category'].apply(lambda x: int(round(budget.loc[x] / 12)))


In [191]:
months['Difference'] = months['Actual Spending'] - months['Budget']

In [195]:
output = months[['Category', 'Actual Spending', 'Budget', 'Difference']]

In [196]:
output

,Category,Actual Spending,Budget,Difference
January,Inventory,341903,191667,150236
February,Inventory,211345,191667,19678
March,Inventory,230989,191667,39322
April,Inventory,263154,191667,71487
May,Inventory,234529,191667,42862
June,Inventory,229251,191667,37584
July,Inventory,236123,191667,44456
August,Inventory,206211,191667,14544
September,Inventory,213083,191667,21416
October,Inventory,207805,191667,16138


## Save Output

In [197]:
output.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/17_monthly_budget.csv')